# 04 — Render the SVG

LOOM's own `transitmap` draws a good map, but the animation needs geometry it
can address: one `<path>` per (line, edge) with a stable id, so a train dot can
be placed with `getPointAtLength()`. So the map is drawn here instead, reusing
LOOM's solved line ordering for the parallel-track offsets.

In [ ]:
%load_ext autoreload
%autoreload 2

from schematic import feeds, loom, pipeline, animate
from schematic.linegraph import LineGraph
from schematic.crs import to_mercator
from schematic.render import render, octilinearity, Style

FEED = "la-metro-rail"
LINE_ORDER = list("ABCDEK")   # the order lines are drawn in, back to front

In [ ]:
from IPython.display import SVG, display

paths = pipeline.schematize(FEED)
graph = LineGraph.from_geojson(paths["octi"]).reproject(to_mercator)
result = render(graph, width=1800, title="LA Metro Rail", line_order=LINE_ORDER)

print(f"{result.width:.0f} x {result.height:.0f}, {len(result.tracks)} track segments")
print(f"labels dropped: {result.dropped_labels}")
display(SVG(result.svg))

### Label placement

A fixed east offset collapses on a horizontal run — the E Line is twenty
stations in a straight row. The placer rotates labels off the line and tests
collisions with oriented boxes, because two 45° labels are thin strips that
slide past each other even though their bounding boxes overlap heavily.

In [ ]:
from collections import Counter
import re
print(Counter(int(m) for m in re.findall(r'rotate\((-?\d+)', result.svg)))
print("haloed (placed over a line):", result.svg.count("paint-order"))

### Cross-check against LOOM's own renderer

Same topology, drawn independently — a useful check that the offsets and
ordering are being read correctly.

In [ ]:
import json

svg = loom.transitmap(json.loads(paths["octi"].read_text()), "-l")
display(SVG(svg))